In [1]:
# === Import Required Libraries ===
import time
import pandas as pd
import os
import sys
import numpy as np
import subprocess

# === Import Local Dependencies ==
#from libraries.readData import readData
from libraries.apiIntegration import loadDataset
from libraries.generateCov import generateCov
from libraries.propagator import monteCarloPropagator, ephemerisPropagator,TLEpropagator
from libraries.orbitAssociation import orbitAssociation
from libraries.stateMetrics import stateMetrics
from libraries.binaryMetrics import binaryMetrics
from libraries.residualMetrics import residualMetrics
from libraries.evaluationReport import evaluationReport
from libraries.generatePDF import generatePDF
import libraries.config as config

In [2]:
loadDataset("./data/output_dataset.json")

(                                         id classificationMarking  \
 0      2826088a-30f5-4201-a29f-a37354a15a91         U//PR-EXO-OBS   
 1      1d236bf6-252f-4f17-8cb6-39dd1dbf74fb         U//PR-EXO-OBS   
 2      f7ee7ed0-5f88-401b-88db-8982ce9d0407         U//PR-EXO-OBS   
 3      f856adcd-d280-486b-9368-8734f52a393e         U//PR-EXO-OBS   
 4      f5fc3100-9c4b-42e4-afd6-592ce3e606e0         U//PR-EXO-OBS   
 ...                                     ...                   ...   
 83658  86998e27-d67f-4df7-a08b-11a5959f2c00         U//PR-EXO-OBS   
 83659  811f0627-7998-44db-ab81-9a7e93ac2000         U//PR-EXO-OBS   
 83660  d41c46ab-ecaa-48b7-814c-0deaefa6a7d7         U//PR-EXO-OBS   
 83661  f7c9995e-b0d1-4754-aec1-5235f63f4219         U//PR-EXO-OBS   
 83662  8e59d222-7af4-47fe-bce8-0e87208be63e         U//PR-EXO-OBS   
 
                                 obTime idSensor taskId origSensorId   uct  \
 0     2025-10-07 16:56:30.673126+00:00  EXO1690      0         1690  True   
 1

In [8]:
input_path = "./data/output_dataset.json"
import json

with open(input_path, 'r') as f:
    data = json.load(f)

# Reconstruct obs_data
obs_data = pd.DataFrame(data['dataset_obs'])
obs_data['obTime'] = pd.to_datetime(obs_data['obTime'],format='mixed')

# Reconstruct track_data
track_data = pd.DataFrame(data['dataset_elset'])

# Reconstruct ref_obs from 'reference' field
reference = pd.DataFrame(data['reference'])

# Reconstruct ref_obs and ref_tracks by correlating groupedObsIds and groupedElsetIds
obs_id_to_satno = {}
elset_id_to_satno = {}
for entry in data["reference"]:
    sat_no = entry["satNo"]
    for obs_id in entry["groupedObsIds"]:
        obs_id_to_satno[obs_id] = sat_no
    for elset_id in entry["groupedElsetIds"]:
        elset_id_to_satno[elset_id] = sat_no

ref_obs = obs_data[obs_data["id"].isin(obs_id_to_satno)].copy()
ref_obs["satNo"] = ref_obs["id"].map(obs_id_to_satno)

ref_track = track_data[track_data["id"].isin(elset_id_to_satno)].copy()
ref_track["satNo"] = ref_track["id"].map(elset_id_to_satno)

# Reconstruct ref_sv
generateCov(reference)
ref_sv = reference[['satNo', 'xpos', 'ypos', 'zpos', 'xvel', 'yvel', 'zvel', 'epoch', 'cov_matrix', 'mass', 'crossSection', 'dragCoeff', 'solarRadPressCoeff']].copy()
ref_sv['epoch'] = pd.to_datetime(ref_sv['epoch'])
#ref_sv['cov_matrix'] = ref_sv['cov_matrix'].apply(lambda x: np.array(json.loads(x)))

# Reconstruct ref_elset
ref_elset = reference[['satNo', 'line1', 'line2']].copy()

In [4]:
#reference['cov']
generateCov(reference)

,satNo,xpos,ypos,zpos,xvel,yvel,zvel,epoch,cov,mass,crossSection,dragCoeff,solarRadPressCoeff,line1,line2,groupedObsIds,groupedElsetIds,cov_matrix
0,19687,39802.274785,-14434.635116,203.879942,1.016757,2.807720,0.704359,2025-10-07 21:12:13.709921,"[0.00023990075455, 0.00035227164566, 0.0013492...",782.95,11.616756,0.0,0.0294,1 19687U 88109A 25280.88349201 +.00000000 +0...,2 19687 13.3421 338.6943 0007752 260.3364 101...,"[a9d21d8d-d730-4f10-ba69-f5bfd41eb8da, 19ee9c8...","[a9d21d8d-d730-4f10-ba69-f5bfd41eb8da, 19ee9c8...","[[0.00023990075455, 0.00035227164566, 5.622733..."
1,38977,30969.899831,-27673.243301,-7321.776460,2.044430,2.295292,-0.039658,2025-10-07 21:12:13.194937,"[0.00013600054465, 0.00011786037737, 0.0002362...",1282.00,24.100000,0.0,0.0268,1 38977U 12061A 25280.88348605 +.00000000 +0...,2 38977 9.9351 52.3131 0007360 164.3214 101...,"[b3c26358-fb57-4336-ac41-b52540772bd0, 5ce1e41...","[b3c26358-fb57-4336-ac41-b52540772bd0, 5ce1e41...","[[0.00013600054465, 0.00011786037737, 2.462270..."
2,28937,10853.053987,-41025.436400,-1859.175908,2.950710,0.788699,-0.246129,2025-10-07 21:11:52.602009,"[0.0005617777086, 0.00013426261715, 4.76023211...",4650.00,57.690667,0.0,0.0257,1 28937U 06004A 25280.88324771 +.00000000 +0...,2 28937 5.1279 76.0705 0011310 153.5871 55...,"[7121e147-bb70-4e84-8b3c-5ff80649b48b, 5576283...","[7121e147-bb70-4e84-8b3c-5ff80649b48b, 5576283...","[[0.0005617777086, 0.00013426261715, -2.745665..."
3,15825,29827.087288,-29516.626730,-3322.414240,2.154175,2.108721,0.624752,2025-10-07 21:11:50.778321,"[0.00034593036925, 0.00027571125912, 0.0003012...",778.00,11.241887,0.0,0.0303,1 15825U 85048C 25280.88322660 +.00000000 +0...,2 15825 12.6488 335.8754 0010076 3.5899 335...,"[485f855a-311b-4958-b621-8b848ff37e78, a89b94f...","[485f855a-311b-4958-b621-8b848ff37e78, a89b94f...","[[0.00034593036925, 0.00027571125912, 6.055934..."
4,4478,30294.608692,-29667.825215,-3278.061668,2.159469,2.118581,-0.123057,2025-10-07 21:11:50.778321,"[0.00025673210457, 0.00025198077775, 0.0002658...",135.78,1.951714,0.0,0.0066,1 04478U 70055A 25280.88322660 +.00000000 +0...,2 04478 4.8372 72.0715 0316873 108.9789 132...,"[9ac4db7c-8c42-45b0-9411-bfa2b58b2f32, 0240840...","[9ac4db7c-8c42-45b0-9411-bfa2b58b2f32, 0240840...","[[0.00025673210457, 0.00025198077775, 9.229295..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
872,27712,34705.052917,23795.271639,1312.841175,-1.718165,2.486513,0.573540,2025-10-07 00:07:46.928548,"[0.00048164334081, -0.0005285488082599999, 0.0...",2775.00,37.969642,0.0,0.0099,1 27712U 03012B 25280.00540427 +.00000000 +0...,2 27712 10.8500 24.7769 0023196 266.6338 103...,"[cabef185-2b01-47ab-97a8-c475152cce16, 9e51622...","[cabef185-2b01-47ab-97a8-c475152cce16, 9e51622...","[[0.00048164334081, -0.0005285488082599999, -0..."
873,35943,10126.983955,40935.719984,-47.753400,-2.984345,0.738038,0.006885,2025-10-06 23:50:14.715546,"[0.0008235127426100001, -0.00010240009946, 7.9...",2440.00,14.112933,0.0,0.0396,1 35943U 09054B 25279.99322587 +.00000000 +0...,2 35943 0.0065 152.5750 0001647 56.9937 226...,"[8b2196f7-a270-4e2a-8fde-0326a29b2028, ab3fa7c...","[8b2196f7-a270-4e2a-8fde-0326a29b2028, ab3fa7c...","[[0.0008235127426100001, -0.00010240009946, 6...."
874,25021,42001.307475,-3397.592897,203.054117,0.240266,2.965233,0.752637,2025-10-06 23:32:34.564025,"[0.00013878487206, 6.2160279171e-05, 0.0017819...",269.57,4.851404,0.0,0.0114,1 25021U 97065C 25279.98095560 +.00000000 +0...,2 25021 14.2280 354.0681 0047478 198.6106 162...,"[f833b692-b010-465e-b8eb-c92654621a19, acfd04e...","[f833b692-b010-465e-b8eb-c92654621a19, acfd04e...","[[0.00013878487206, 6.2160279171e-05, 1.166157..."
875,41588,6778.700548,41615.323656,-32.002176,-3.034773,0.494049,0.006895,2025-10-06 23:21:59.902571,"[0.0028009442819, 0.0040876611048, 0.013672535...",1954.00,33.226533,0.0,0.0392,1 41588U 16038A 25279.97360998 +.00000000 +0...,2 41588 0.0459 25.8841 0001395 118.8778 296...,"[f7ec0114-2c1e-4529-a3e4-4e24

In [5]:
reference['cov_matrix']

0      [[0.00023990075455, 0.00035227164566, 5.622733...
1      [[0.00013600054465, 0.00011786037737, 2.462270...
2      [[0.0005617777086, 0.00013426261715, -2.745665...
3      [[0.00034593036925, 0.00027571125912, 6.055934...
4      [[0.00025673210457, 0.00025198077775, 9.229295...
                             ...                        
872    [[0.00048164334081, -0.0005285488082599999, -0...
873    [[0.0008235127426100001, -0.00010240009946, 6....
874    [[0.00013878487206, 6.2160279171e-05, 1.166157...
875    [[0.0028009442819, 0.0040876611048, 9.64526240...
876    [[0.00037848370395, -0.00045625510649, -8.5524...
Name: cov_matrix, Length: 877, dtype: object

In [8]:
import ast

def find_bad_strings(x):
    """Returns True if x is a string that literal_eval fails on."""
    if not isinstance(x, str):
        # It's not a string, so it's not the problem
        return False
    
    try:
        ast.literal_eval(x)
        # It's a valid literal string (e.g., "[1, 2]", "None")
        return False
    except (ValueError, SyntaxError):
        # This is a bad string (e.g., "nan", "inf", "hello")
        return True

# --- DEBUGGING CODE ---
# Apply this function to create a boolean mask
problematic_mask = reference['cov'].apply(find_bad_strings)

# Use the mask to select and print the problematic entries
bad_entries = reference['cov'][problematic_mask]

if not bad_entries.empty:
    print("Found these non-literal-eval-able strings in 'cov' column:")
    # .unique() is helpful if there are thousands of "nan" strings
    print(bad_entries.unique()) 
else:
    print("No problematic strings found in 'cov' column.")
# --- END DEBUGGING ---

Found these non-literal-eval-able strings in 'cov' column:
['NaN']


In [ ]:
evals = evaluationReport(association_results, binary_results, state_results, residual_ref_results, residual_cand_results, './data/raw_results.json')
